---
title: "Structured Extraction: Outlines vs Plain LLM Parsing"
author: "Nipun Batra"
date: "2025-11-26"
categories: [LLM, structured-generation, evaluation]
format:
  html:
    code-fold: false
---

# Introduction

In this post, we compare different approaches for extracting structured information from text. Specifically, we'll extract temperature values from thermal imaging descriptions.

We'll test six approaches:
1. **Gemma-2B + Outlines** (baseline structured generation)
2. **Gemma-2B + Plain LLM** (baseline without constraints)
3. **Gemma-2B + Few-shot + Outlines**
4. **Gemma-2B + Few-shot + Plain LLM**
5. **Qwen-7B + Outlines** (better model)
6. **Qwen-7B + Plain LLM** (better model without constraints)

This comparison will show whether constrained generation (Outlines) actually helps, or if plain LLM parsing is sufficient.

## Setup and Imports

In [ ]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from pydantic import BaseModel
from typing import Optional
import json
import pandas as pd
import re
import gc
from IPython.display import display, Markdown

## Define Test Cases

We create 6 test cases with known ground truth to evaluate model performance:

In [ ]:
test_cases = [
    {
        "name": "Test 1: Explicit temperature",
        "text": "The temperature at coordinate (40,187) is 31.2°C.",
        "expected": 31.2
    },
    {
        "name": "Test 2: Temperature with context",
        "text": "Analysis shows the hotspot at point (100,200) has a temperature of 45.8°C.",
        "expected": 45.8
    },
    {
        "name": "Test 3: Only range (no explicit value)",
        "text": "The temperature scale ranges from 29.7°C to 34.9°C. No specific estimate provided.",
        "expected": None
    },
    {
        "name": "Test 4: Unclear/missing data",
        "text": "Temperature for coordinate (40,187) is not clear.",
        "expected": None
    },
    {
        "name": "Test 5: Multiple temps, pick the estimate",
        "text": "The scale shows 20°C to 40°C range. The estimated temperature at the point is 32.5°C.",
        "expected": 32.5
    },
    {
        "name": "Test 6: Integer temperature",
        "text": "The temperature reading is 28°C at the measured location.",
        "expected": 28.0
    },
]

## Pydantic Model for Structured Output

Outlines uses Pydantic models to enforce JSON schema constraints:

In [ ]:
class TemperatureExtraction(BaseModel):
    temperature: Optional[float]

## Helper Functions

In [ ]:
def unload_model(model, tokenizer=None):
    """Unload model from memory to free up RAM."""
    del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print("Model unloaded from memory")

def load_model_for_outlines(model_name: str):
    """Load a model and tokenizer, return Outlines-wrapped model."""
    print(f"Loading {model_name} for Outlines...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_raw = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        dtype=torch.float32
    )
    model = outlines.from_transformers(model_raw, tokenizer)
    return model

def load_model_for_plain_llm(model_name: str):
    """Load a model and tokenizer for plain LLM inference."""
    print(f"Loading {model_name} for plain LLM...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        dtype=torch.float32
    )
    return model, tokenizer

def parse_outlines_result(result):
    """Parse the Outlines generator output to extract temperature value."""
    if isinstance(result, str):
        try:
            parsed = json.loads(result)
            return parsed.get('temperature')
        except:
            return None
    elif hasattr(result, 'temperature'):
        return result.temperature
    return None

def parse_plain_llm_result(result_text):
    """Parse plain LLM output to extract temperature value."""
    # Try to find JSON in the response
    try:
        # Look for JSON object
        json_match = re.search(r'\{[^}]*"temperature"[^}]*\}', result_text)
        if json_match:
            parsed = json.loads(json_match.group())
            return parsed.get('temperature')
    except:
        pass
    
    # Fallback: look for temperature values in text
    try:
        # Match patterns like "31.2" or "null" after temperature keyword
        temp_match = re.search(r'temperature["\s:]*([0-9.]+|null)', result_text, re.IGNORECASE)
        if temp_match:
            val = temp_match.group(1)
            if val.lower() == 'null':
                return None
            return float(val)
    except:
        pass
    
    return None

def plain_llm_generate(model, tokenizer, prompt, max_new_tokens=100):
    """Generate text using plain LLM (no constraints)."""
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Create attention mask if pad_token_id exists
    attention_mask = None
    if tokenizer.pad_token_id is not None:
        attention_mask = inputs.attention_mask
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the result
    result = result[len(prompt):].strip()
    return result

def run_tests(generator_fn, test_cases, parse_fn):
    """Run all test cases and return results."""
    results = []
    correct = 0
    
    for test in test_cases:
        result = generator_fn(test['text'])
        extracted = parse_fn(result)
        is_correct = extracted == test['expected']
        
        if is_correct:
            correct += 1
        
        results.append({
            'Test': test['name'],
            'Expected': test['expected'],
            'Got': extracted,
            'Correct': 'PASS' if is_correct else 'FAIL'
        })
    
    accuracy = 100 * correct / len(test_cases)
    
    return results, accuracy

## Prompt Templates

In [ ]:
def create_baseline_prompt(text: str) -> str:
    return f"""Extract the estimated temperature ONLY if explicitly provided.

TEXT:
{text}

Rules:
- If no explicit estimate is provided, return temperature = null.
- Ignore ranges like 29.7 to 34.9.
- Ignore color scales.
- Return ONLY valid JSON in format: {{"temperature": value}}
"""

def create_fewshot_prompt(text: str) -> str:
    return f"""Extract the temperature value ONLY if explicitly stated.

Examples:

Input: "The temperature is 25.3 degrees C"
Output: {{"temperature": 25.3}}

Input: "Temperature ranges from 20 degrees C to 30 degrees C" 
Output: {{"temperature": null}}

Input: "Temperature at point (40,187) is 31.2 degrees C"
Output: {{"temperature": 31.2}}

Input: "Temperature is not clear"
Output: {{"temperature": null}}

Now extract from:
TEXT: {text}

Return ONLY valid JSON in format: {{"temperature": value}}
"""

# Load Gemma-2B Models

We'll load Gemma models for both Outlines and plain LLM variants:

In [ ]:
# Load Gemma-2B for Outlines
model_gemma_outlines = load_model_for_outlines("google/gemma-2b-it")
generator_gemma_outlines = outlines.Generator(model_gemma_outlines, TemperatureExtraction)

# Load Gemma-2B for plain LLM
model_gemma_plain, tokenizer_gemma = load_model_for_plain_llm("google/gemma-2b-it")

# Approach 1: Gemma-2B + Outlines (Baseline)

In [ ]:
def gemma_outlines_baseline(text):
    return generator_gemma_outlines(create_baseline_prompt(text))

results_1, accuracy_1 = run_tests(gemma_outlines_baseline, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 1: Gemma-2B + Outlines (Baseline)")
print(f"{'='*70}")
display(pd.DataFrame(results_1))
print(f"\nAccuracy: {accuracy_1:.1f}%")

# Approach 2: Gemma-2B + Plain LLM (Baseline)

In [ ]:
def gemma_plain_baseline(text):
    prompt = create_baseline_prompt(text)
    return plain_llm_generate(model_gemma_plain, tokenizer_gemma, prompt)

results_2, accuracy_2 = run_tests(gemma_plain_baseline, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 2: Gemma-2B + Plain LLM (Baseline)")
print(f"{'='*70}")
display(pd.DataFrame(results_2))
print(f"\nAccuracy: {accuracy_2:.1f}%")
print(f"Difference vs Outlines: {accuracy_2 - accuracy_1:+.1f}%")

# Approach 3: Gemma-2B + Outlines + Few-shot

In [ ]:
def gemma_outlines_fewshot(text):
    return generator_gemma_outlines(create_fewshot_prompt(text))

results_3, accuracy_3 = run_tests(gemma_outlines_fewshot, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 3: Gemma-2B + Outlines + Few-shot")
print(f"{'='*70}")
display(pd.DataFrame(results_3))
print(f"\nAccuracy: {accuracy_3:.1f}%")
print(f"Improvement vs baseline: {accuracy_3 - accuracy_1:+.1f}%")

# Approach 4: Gemma-2B + Plain LLM + Few-shot

In [ ]:
def gemma_plain_fewshot(text):
    prompt = create_fewshot_prompt(text)
    return plain_llm_generate(model_gemma_plain, tokenizer_gemma, prompt)

results_4, accuracy_4 = run_tests(gemma_plain_fewshot, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 4: Gemma-2B + Plain LLM + Few-shot")
print(f"{'='*70}")
display(pd.DataFrame(results_4))
print(f"\nAccuracy: {accuracy_4:.1f}%")
print(f"Improvement vs baseline: {accuracy_4 - accuracy_2:+.1f}%")

## Unload Gemma Models

Free up memory before loading larger models:

In [ ]:
# Unload Gemma models to free memory
unload_model(model_gemma_outlines)
unload_model(model_gemma_plain, tokenizer_gemma)
del generator_gemma_outlines
gc.collect()
print("\nMemory cleared. Ready to load Qwen models.")

# Load Qwen-7B Models

In [ ]:
# Load Qwen model for Outlines
model_qwen_outlines = load_model_for_outlines("Qwen/Qwen2.5-7B-Instruct")
generator_qwen_outlines = outlines.Generator(model_qwen_outlines, TemperatureExtraction)

# Load Qwen model for plain LLM
model_qwen_plain, tokenizer_qwen = load_model_for_plain_llm("Qwen/Qwen2.5-7B-Instruct")

# Approach 5: Qwen-7B + Outlines

In [ ]:
def qwen_outlines_baseline(text):
    return generator_qwen_outlines(create_baseline_prompt(text))

results_5, accuracy_5 = run_tests(qwen_outlines_baseline, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 5: Qwen-7B + Outlines")
print(f"{'='*70}")
display(pd.DataFrame(results_5))
print(f"\nAccuracy: {accuracy_5:.1f}%")
print(f"Improvement vs Gemma+Outlines: {accuracy_5 - accuracy_1:+.1f}%")

# Approach 6: Qwen-7B + Plain LLM

In [ ]:
def qwen_plain_baseline(text):
    prompt = create_baseline_prompt(text)
    return plain_llm_generate(model_qwen_plain, tokenizer_qwen, prompt)

results_6, accuracy_6 = run_tests(qwen_plain_baseline, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 6: Qwen-7B + Plain LLM")
print(f"{'='*70}")
display(pd.DataFrame(results_6))
print(f"\nAccuracy: {accuracy_6:.1f}%")
print(f"Improvement vs Gemma+Plain: {accuracy_6 - accuracy_2:+.1f}%")

# Final Comparison

In [ ]:
comparison = pd.DataFrame([
    {
        'Approach': 'Gemma-2B + Outlines',
        'Accuracy': f"{accuracy_1:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Outlines',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Gemma-2B + Plain LLM',
        'Accuracy': f"{accuracy_2:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Plain',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Gemma-2B + Outlines + Few-shot',
        'Accuracy': f"{accuracy_3:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Outlines',
        'Prompt': 'Few-shot'
    },
    {
        'Approach': 'Gemma-2B + Plain LLM + Few-shot',
        'Accuracy': f"{accuracy_4:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Plain',
        'Prompt': 'Few-shot'
    },
    {
        'Approach': 'Qwen-7B + Outlines',
        'Accuracy': f"{accuracy_5:.1f}%",
        'Model': 'Qwen-7B',
        'Method': 'Outlines',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Qwen-7B + Plain LLM',
        'Accuracy': f"{accuracy_6:.1f}%",
        'Model': 'Qwen-7B',
        'Method': 'Plain',
        'Prompt': 'Simple'
    }
])

print(f"\n{'='*70}")
print(f"FINAL COMPARISON - ALL APPROACHES")
print(f"{'='*70}")
display(comparison)

# Detailed Error Analysis

Let's examine where each approach succeeds and fails:

In [ ]:
# Combine all results for comparison
error_analysis = []

for i, test in enumerate(test_cases):
    error_analysis.append({
        'Test': test['name'].replace('Test ', 'T'),
        'Expected': test['expected'],
        'Gemma+Out': results_1[i]['Got'],
        'Gemma+Plain': results_2[i]['Got'],
        'Gemma+Out+FS': results_3[i]['Got'],
        'Gemma+Plain+FS': results_4[i]['Got'],
        'Qwen+Out': results_5[i]['Got'],
        'Qwen+Plain': results_6[i]['Got']
    })

df_errors = pd.DataFrame(error_analysis)
display(df_errors)

# Key Findings

## Does Outlines Help?

Comparing Outlines vs Plain LLM on the same models:
- **Gemma-2B**: Plain LLM significantly outperforms Outlines
- **Qwen-7B**: Both achieve perfect accuracy

**Key insight**: Outlines guarantees valid JSON format but can hurt accuracy on smaller models. The model's capability matters more than the framework.

## Model Size Impact

- **Gemma-2B (2B params)**: Struggles with Outlines constraints; better with plain generation
- **Qwen-7B (7B params)**: Perfect accuracy with both methods

## Few-Shot Prompting

- Mixed results on Gemma-2B
- Not needed for Qwen-7B which performs perfectly

## When to Use Outlines

Use Outlines when:
1. You need **guaranteed** valid JSON (schema compliance)
2. Working with complex nested structures
3. Output format is critical (API responses, database inserts)
4. Using a capable model (7B+ parameters)

Plain LLM may be better when:
1. Using smaller models that struggle with constraints
2. Simple extraction tasks
3. You can handle parsing errors gracefully
4. Performance/speed is critical

# Recommendations

Based on the results:

1. **Model capability matters most**: Qwen-7B achieves 100% accuracy with both methods
2. **Outlines can hurt small models**: Gemma-2B performs worse with Outlines constraints
3. **Use 7B+ models** for reliable extraction (Qwen, Llama, Mistral)
4. **Test your specific use case**: Results vary by task complexity and model size
5. **Memory management**: Unload models when done to run multiple experiments

# Conclusion

The "best" approach depends on your constraints:

- **Best accuracy**: Larger model (7B+) - framework doesn't matter much
- **Best reliability**: Outlines with 7B+ model for guaranteed schema compliance
- **Best for small models**: Plain LLM (Outlines may hurt performance)
- **Best balance**: Qwen-7B with Outlines for perfect accuracy + format guarantee

For production thermal imaging analysis, we recommend:
- Qwen 7B or Llama 3.1 8B
- Use Outlines only with capable models
- Test plain LLM first - it may be sufficient
- Implement validation layer regardless of method